# CSV-Checker für Touch-Scrolling-Ergebnisse

Dieses Notebook liest CSV-Dateien dieses Formats ein, bereitet die Daten auf und prüft, wer alle Durchläufe bzw. alle erforderlichen Resultate abgeschlossen hat.

Es erwartet Spalten wie `id`, `attempts`, `currentParameterSet`, `nextParameterSet`, `parameterBlockMetrics`, `gender`, `group`, `handedness` und `smartphone`.

In [12]:
import json
import math
import re
from pathlib import Path

import pandas as pd

CSV_PATH = Path(r"C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\csv_results\alle_results.csv")

df = pd.read_csv(CSV_PATH)
print(f"{len(df)} Zeilen, {len(df.columns)} Spalten geladen aus {CSV_PATH}")
df.head()

247 Zeilen, 12 Spalten geladen aus C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\csv_results\alle_results.csv


,id,attempts,createdAt,currentParameterSet,gender,group,handedness,nextParameterSet,parameterBlockMetrics,prolificPid,smartphone,updatedAt
0,b0b63c19-2a58-4d5f-b847-b34b881a46fb,"[{""M"":{""parameterSet"":{""M"":{""fingerVelocityPxM...",2026-10-06T09:18:32.744Z,"{""generatedFromAttemptCount"":{""N"":""0""},""scroll...",male,1,right,"{""source"":{""S"":""terraform-appsync-sagemaker-ac...","[{""M"":{""model"":{""M"":{""trainingRowCount"":{""N"":""...",6932ba422722a936d9dc7159,iPhone 16 Pro,2026-10-06T09:41:14.220Z
1,e0749587-c16d-4027-9daa-62392a628675,"[{""M"":{""runNumber"":{""N"":""1""},""totalNormalizedT...",2026-10-06T08:48:21.733Z,"{""inflexion"":{""N"":""0.23""},""status"":{""S"":""ready...",male,2,right,"{""generation"":{""M"":{""strategy"":{""S"":""qLogNoisy...","[{""M"":{""model"":{""M"":{""totalBlockObservations"":...",663cc9dba1feafa5a757e8ea,iPhone 15,2026-10-06T09:01:52.740Z
2,215cd97d-14f4-48a5-9031-f2e751d1b360,"[{""M"":{""parameterSet"":{""M"":{""decelerationRateI...",2026-10-06T09:53:21.510Z,"{""generatedFromAttemptCount"":{""N"":""0""},""decele...",male,2,right,"{""generatedFromAttemptCount"":{""N"":""150""},""gene...","[{""M"":{""model"":{""M"":{""objectiveType"":{""S"":""tot...",699303e736ab87385a91f333,iPhone 15,2026-10-06T10:15:59.517Z
3,27f552c9-c590-4329-a2f9-9c45eb0b0ae3,[],2026-10-06T09:15:13.529Z,"{""status"":{""S"":""ready""},""completedBlockCount"":...",male,1,right,NaN,NaN,69fdf6a169fbc9101f3adeb3,iPhone 15 Pro,2026-10-06T09:15:13.529Z
4,7194ce45-f67c-4e35-b2fa-dec79f6f500f,"[{""M"":{""totalNormalizedTimeMs"":{""N"":""459.84641...",2026-10-06T09:09:23.915Z,"{""decelerationRate"":{""N"":""8.6403""},""generatedF...",male,2,right,"{""status"":{""S"":""ready""},""scrollFriction"":{""N"":...","[{""M"":{""trainingObservationCount"":{""N"":""3""},""g...",69a5dfa3f587ca27e1caf19d,iPhone 16,2026-10-06T09:36:44.944Z


In [13]:
import json

REQUIRED_BLOCKS = 20  # Bloecke pro Teilnehmer
BLOCK_SIZE = 10  # Versuche pro Block


def dynamo_to_py(value):
    """Wandelt DynamoDB-JSON ({"M": ...}, {"L": ...}, {"N": ...}, {"S": ...}) in Python-Objekte um."""
    (type_, inner), = value.items()
    if type_ == "M":
        return {k: dynamo_to_py(v) for k, v in inner.items()}
    if type_ == "L":
        return [dynamo_to_py(v) for v in inner]
    if type_ == "N":
        return int(inner) if inner.lstrip("-").isdigit() else float(inner)
    if type_ == "BOOL":
        return bool(inner)
    if type_ == "NULL":
        return None
    return inner


def completed_blocks_meta(next_parameter_set):
    try:
        return int(dynamo_to_py({"M": json.loads(next_parameter_set)})["completedBlockCount"])
    except (TypeError, ValueError, KeyError):
        return None


df["blocks"] = df["attempts"].apply(lambda s: [dynamo_to_py(b) for b in json.loads(s)])

attempts_summary = pd.DataFrame({
    "id": df["id"],
    "prolificPid": df["prolificPid"],
    "group": df["group"],
    "n_blocks": df["blocks"].apply(len),
    "n_attempts": df["blocks"].apply(lambda bs: sum(len(b["attempts"]) for b in bs)),
    "blocks_complete": df["blocks"].apply(lambda bs: all(len(b["attempts"]) == BLOCK_SIZE for b in bs)),
    "run_numbers_ok": df["blocks"].apply(lambda bs: [b["runNumber"] for b in bs] == list(range(1, len(bs) + 1))),
    "blocks_meta": df["nextParameterSet"].apply(completed_blocks_meta),
})
attempts_summary["meta_diff"] = attempts_summary["blocks_meta"] - attempts_summary["n_blocks"]


def status(n_blocks):
    if n_blocks == 0:
        return "keine Versuche"
    return "vollstaendig" if n_blocks >= REQUIRED_BLOCKS else "unvollstaendig"


attempts_summary["status"] = attempts_summary["n_blocks"].apply(status)

print("Teilnehmer gesamt:", len(attempts_summary))
print("\nStatus:")
print(attempts_summary["status"].value_counts().to_string())
print("\nStatus je Gruppe:")
print(pd.crosstab(attempts_summary["group"], attempts_summary["status"]).to_string())

with_attempts = attempts_summary[attempts_summary["n_blocks"] > 0]
print("\nBloecke pro Teilnehmer (nur mit Versuchen):")
print(with_attempts["n_blocks"].describe().round(2).to_string())
print("\nVerteilung der Blockanzahl:")
print(with_attempts["n_blocks"].value_counts().sort_index().to_string())
print("\nVersuche gesamt:", attempts_summary["n_attempts"].sum())

problems = with_attempts[~(with_attempts["blocks_complete"] & with_attempts["run_numbers_ok"])]
print(f"\nInkonsistenzen (Blockgroesse oder runNumber): {len(problems)}")
if len(problems):
    display(problems)

print("\nAbweichung completedBlockCount (nextParameterSet) - gespeicherte Bloecke:")
print(with_attempts["meta_diff"].value_counts(dropna=False).sort_index().to_string())

attempts_summary.sort_values("n_blocks", ascending=False).head(10)

Teilnehmer gesamt: 247

Status:
status
keine Versuche    209
unvollstaendig     28
vollstaendig       10

Status je Gruppe:
status  keine Versuche  unvollstaendig  vollstaendig
group                                               
1                   69              13             7
2                  140              15             3

Bloecke pro Teilnehmer (nur mit Versuchen):
count    38.00
mean     10.92
std       6.91
min       1.00
25%       5.25
50%       8.50
75%      19.75
max      20.00

Verteilung der Blockanzahl:
n_blocks
1      2
2      1
3      3
4      2
5      2
6      3
7      3
8      3
9      2
10     1
13     2
15     2
18     1
19     1
20    10

Versuche gesamt: 4150

Inkonsistenzen (Blockgroesse oder runNumber): 0

Abweichung completedBlockCount (nextParameterSet) - gespeicherte Bloecke:
meta_diff
-1.0    11
 0.0    24
 NaN     3


,id,prolificPid,group,n_blocks,n_attempts,blocks_complete,run_numbers_ok,blocks_meta,meta_diff,status
0,b0b63c19-2a58-4d5f-b847-b34b881a46fb,6932ba422722a936d9dc7159,1,20,200,True,True,19.0,-1.0,vollstaendig
5,f2003946-965e-4a66-a438-60734e99ccb2,698a8b3d9811cfd9e748450e,2,20,200,True,True,19.0,-1.0,vollstaendig
4,7194ce45-f67c-4e35-b2fa-dec79f6f500f,69a5dfa3f587ca27e1caf19d,2,20,200,True,True,19.0,-1.0,vollstaendig
6,050f3f9b-f699-4175-a200-6378fba85425,6a9c95b942aecb1a5e16ca69,1,20,200,True,True,19.0,-1.0,vollstaendig
10,2384fbcc-826c-402f-8069-08f92495bea6,698354c40e8075e4c4602fbf,1,20,200,True,True,19.0,-1.0,vollstaendig
8,313de046-9eb8-4fd6-afcc-4f3ca00fdad1,6a33b6f5e3d9dff938626d9d,1,20,200,True,True,19.0,-1.0,vollstaendig
7,5fdb6967-2279-4113-ba9e-4643015d1858,662121d5f125febc8db80f74,1,20,200,True,True,19.0,-1.0,vollstaendig
20,c1404ef8-6b67-4e8e-8060-fb93d9ca7187,6447bd2acb4eeecd9451dbbe,2,20,200,True,True,19.0,-1.0,vollstaendig
12,c0b77808-319e-4595-99ea-8b0627ce3f2e,6a29409ecb14cd57b7d98746,1,20,200,True,True,19.0,-1.0,vollstaendig
42,ea07ded7-91ef-401c-910a-94a66cfab9e2,6a968836c2a369c9285714da,1,20,200,True,True,19.0,-1.0,vollstaendig


In [14]:
empty = attempts_summary[attempts_summary["n_attempts"] == 0].copy()
empty = empty.join(df[["createdAt", "updatedAt", "nextParameterSet", "parameterBlockMetrics"]])

print(f"Eintraege ohne Versuche: {len(empty)} von {len(df)}")
print("Max. Versuche in diesen Eintraegen:", empty["n_attempts"].max())
print("\nGruppe:")
print(empty["group"].value_counts().sort_index().to_string())
print("\nNie aktualisiert (createdAt == updatedAt):", (empty["createdAt"] == empty["updatedAt"]).sum())
print("Mit nextParameterSet / parameterBlockMetrics:", empty["nextParameterSet"].notna().sum(), "/", empty["parameterBlockMetrics"].notna().sum())
print("\nErstellt pro Stunde (UTC):")
print(pd.to_datetime(empty["createdAt"]).dt.floor("h").value_counts().sort_index().to_string())

# Ein Prolific-Teilnehmer kann mehrere Eintraege haben (z. B. Neustart der Studie)
per_pid = (
    df.assign(n_attempts=attempts_summary["n_attempts"], n_blocks=attempts_summary["n_blocks"])
    .groupby("prolificPid")
    .agg(entries=("id", "count"), empty_entries=("n_attempts", lambda s: (s == 0).sum()),
         total_attempts=("n_attempts", "sum"), total_blocks=("n_blocks", "sum"), max_blocks=("n_blocks", "max"))
)
only_empty = per_pid[per_pid["total_attempts"] == 0]
also_other = per_pid[(per_pid["empty_entries"] > 0) & (per_pid["total_attempts"] > 0)]

print(f"\nProlific-IDs gesamt: {len(per_pid)}")
print(f"  nur leere Eintraege (0 Versuche insgesamt): {len(only_empty)}")
print(f"  leere Eintraege, aber auch Eintrag mit Versuchen: {len(also_other)}")
print(f"  nur Eintraege mit Versuchen: {len(per_pid) - len(only_empty) - len(also_other)}")
print("\nLeere Eintraege pro Prolific-ID:")
print(per_pid.loc[per_pid["empty_entries"] > 0, "empty_entries"].describe().round(2).to_string())

print("\nProlific-IDs mit leeren Eintraegen UND Versuchen in anderem Eintrag:")
display(also_other.sort_values("total_blocks", ascending=False))

only_empty.sort_values("entries", ascending=False).head(10)

Eintraege ohne Versuche: 209 von 247
Max. Versuche in diesen Eintraegen: 0

Gruppe:
group
1     69
2    140

Nie aktualisiert (createdAt == updatedAt): 209
Mit nextParameterSet / parameterBlockMetrics: 0 / 0

Erstellt pro Stunde (UTC):
createdAt
2026-10-06 08:00:00+00:00     26
2026-10-06 09:00:00+00:00    130
2026-10-06 10:00:00+00:00     33
2026-10-06 11:00:00+00:00     20

Prolific-IDs gesamt: 108
  nur leere Eintraege (0 Versuche insgesamt): 73
  leere Eintraege, aber auch Eintrag mit Versuchen: 16
  nur Eintraege mit Versuchen: 19

Leere Eintraege pro Prolific-ID:
count    89.00
mean      2.35
std       1.29
min       1.00
25%       2.00
50%       2.00
75%       3.00
max       9.00

Prolific-IDs mit leeren Eintraegen UND Versuchen in anderem Eintrag:


,entries,empty_entries,total_attempts,total_blocks,max_blocks
prolificPid,,,,,
698a8b3d9811cfd9e748450e,3,1,250,25,20
6932ba422722a936d9dc7159,2,1,200,20,20
6a1c005498e4776f80f8b138,2,1,180,18,18
67341b607b7ab78bc222464e,4,2,140,14,8
5e92267dcc3d6c553eb8c817,2,1,90,9,9
665c7c91897b3e6380ffb2e7,2,1,80,8,8
6aab9268585125ed56fcde1b,5,4,80,8,8
69e213021f9224d147d5908b,2,1,70,7,7
6a7cbbd6e8fda6928d774938,2,1,70,7,7


,entries,empty_entries,total_attempts,total_blocks,max_blocks
prolificPid,,,,,
6a0f70cf06a88ee3e7ae2235,9,9,0,0,0
697b25947b6c942ad6ad98a0,6,6,0,0,0
69f9aff0d2852ed04bf7ae3c,5,5,0,0,0
697ccc1627806e7b58b7201f,5,5,0,0,0
6a00e36e3f9414f0d94977fd,5,5,0,0,0
697b09e893ebfe42fbdbbaae,4,4,0,0,0
6982523e224b28f05388f724,4,4,0,0,0
6987229edb16fc00464c0b03,4,4,0,0,0
69ff28e6a69c08d18bf39e87,4,4,0,0,0


In [15]:
# Prolific-Export: Teilnehmer mit Status COMPLETED (Reihenfolge wie im Export)
PROLIFIC_COMPLETED = [
    "6a968836c2a369c9285714da",
    "698a8b3d9811cfd9e748450e",
    "6447bd2acb4eeecd9451dbbe",
    "67f789ca382e36a759a011af",
    "698354c40e8075e4c4602fbf",
    "662121d5f125febc8db80f74",
    "69a5dfa3f587ca27e1caf19d",
    "6932ba422722a936d9dc7159",
    "6a33b6f5e3d9dff938626d9d",
    "6a9c95b942aecb1a5e16ca69",
]

# Pro Prolific-ID den Eintrag mit den meisten Bloecken waehlen (bei Gleichstand den zuletzt aktualisierten)
entries = attempts_summary.join(df[["updatedAt"]])
best_entry = (
    entries[entries["prolificPid"].isin(PROLIFIC_COMPLETED)]
    .sort_values(["n_blocks", "updatedAt"], ascending=False)
    .drop_duplicates("prolificPid")
    .set_index("prolificPid")
    .reindex(PROLIFIC_COMPLETED)
)
best_entry["entries_total"] = entries.groupby("prolificPid").size().reindex(PROLIFIC_COMPLETED)

print("Prolific-IDs in Liste:", len(PROLIFIC_COMPLETED), "| nicht in CSV gefunden:", best_entry["id"].isna().sum())
print("Davon mit 20 Bloecken:", (best_entry["n_blocks"] >= REQUIRED_BLOCKS).sum())

csv_complete = set(entries.loc[entries["status"] == "vollstaendig", "prolificPid"])
print("\nLaut Prolific COMPLETED, aber in der CSV nicht komplett:", sorted(set(PROLIFIC_COMPLETED) - csv_complete))
print("In der CSV komplett, aber nicht in der Prolific-Liste:", sorted(csv_complete - set(PROLIFIC_COMPLETED)))

best_entry[["id", "group", "entries_total", "n_blocks", "n_attempts", "status"]]

Prolific-IDs in Liste: 10 | nicht in CSV gefunden: 0
Davon mit 20 Bloecken: 9

Laut Prolific COMPLETED, aber in der CSV nicht komplett: ['67f789ca382e36a759a011af']
In der CSV komplett, aber nicht in der Prolific-Liste: ['6a29409ecb14cd57b7d98746']


,id,group,entries_total,n_blocks,n_attempts,status
prolificPid,,,,,,
6a968836c2a369c9285714da,ea07ded7-91ef-401c-910a-94a66cfab9e2,1,1,20,200,vollstaendig
698a8b3d9811cfd9e748450e,f2003946-965e-4a66-a438-60734e99ccb2,2,3,20,200,vollstaendig
6447bd2acb4eeecd9451dbbe,c1404ef8-6b67-4e8e-8060-fb93d9ca7187,2,1,20,200,vollstaendig
67f789ca382e36a759a011af,3a17c7b9-8bf4-4c35-9459-55e815318199,1,3,3,30,unvollstaendig
698354c40e8075e4c4602fbf,2384fbcc-826c-402f-8069-08f92495bea6,1,1,20,200,vollstaendig
662121d5f125febc8db80f74,5fdb6967-2279-4113-ba9e-4643015d1858,1,1,20,200,vollstaendig
69a5dfa3f587ca27e1caf19d,7194ce45-f67c-4e35-b2fa-dec79f6f500f,2,1,20,200,vollstaendig
6932ba422722a936d9dc7159,b0b63c19-2a58-4d5f-b847-b34b881a46fb,1,2,20,200,vollstaendig
6a33b6f5e3d9dff938626d9d,313de046-9eb8-4fd6-afcc-4f3ca00fdad1,1,1,20,200,vollstaendig


In [16]:
OUTPUT_DIR = CSV_PATH.parent / "aufbereitet"
OUTPUT_DIR.mkdir(exist_ok=True)


def prepare_participant(row):
    """Baut aus einem CSV-Eintrag die Struktur der JSON-Exporte in ergebnisse/."""
    blocks = sorted(row["blocks"], key=lambda b: b["runNumber"])
    attempt_blocks = [
        {"runNumber": b["runNumber"], "parameterSet": b["parameterSet"], "attempts": b["attempts"]}
        for b in blocks
    ]
    flat_attempts = [
        {**a, "blockIndex": b["runNumber"], "blockParameterSet": b["parameterSet"]}
        for b in attempt_blocks
        for a in sorted(b["attempts"], key=lambda a: a["attemptInBlock"])
    ]
    return {
        "participantId": row["id"],
        "prolificPid": row["prolificPid"],
        "gender": row["gender"],
        "smartphone": row["smartphone"],
        "handedness": row["handedness"],
        "group": int(row["group"]),
        "currentParameterSet": dynamo_to_py({"M": json.loads(row["currentParameterSet"])}),
        "nextParameterSet": dynamo_to_py({"M": json.loads(row["nextParameterSet"])}) if pd.notna(row["nextParameterSet"]) else None,
        "finalParameterSet": None,
        "parameterBlockMetrics": dynamo_to_py({"L": json.loads(row["parameterBlockMetrics"])}) if pd.notna(row["parameterBlockMetrics"]) else [],
        "attemptBlocks": attempt_blocks,
        "attempts": flat_attempts,
    }


df_by_id = df.set_index("id")
prepared = {}
for pid in PROLIFIC_COMPLETED:
    entry_id = best_entry.loc[pid, "id"]
    participant = prepare_participant(df_by_id.loc[entry_id].to_dict() | {"id": entry_id, "blocks": df.loc[df["id"] == entry_id, "blocks"].iloc[0]})
    prepared[pid] = participant
    date = df_by_id.loc[entry_id, "createdAt"][:10]
    with open(OUTPUT_DIR / f"touch-scrolling-data-{pid}-{date}.json", "w", encoding="utf-8") as fh:
        json.dump(participant, fh, ensure_ascii=False, indent=2)

overview = pd.DataFrame([
    {
        "prolificPid": pid,
        "group": p["group"],
        "blocks": len(p["attemptBlocks"]),
        "attempts": len(p["attempts"]),
        "flicks": sum(len(a["flicks"]) for a in p["attempts"]),
        "block_metrics": len(p["parameterBlockMetrics"]),
        "mean_timeMs": round(pd.Series([a["timeMs"] for a in p["attempts"]]).mean(), 1),
        "overshoot_rate": round(pd.Series([a["didOvershoot"] for a in p["attempts"]]).mean(), 3),
    }
    for pid, p in prepared.items()
])
print(f"{len(prepared)} Teilnehmer aufbereitet und gespeichert in {OUTPUT_DIR}")
overview

10 Teilnehmer aufbereitet und gespeichert in C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\csv_results\aufbereitet


,prolificPid,group,blocks,attempts,flicks,block_metrics,mean_timeMs,overshoot_rate
0,6a968836c2a369c9285714da,1,20,200,2540,17,4675.7,0.530
1,698a8b3d9811cfd9e748450e,2,20,200,1534,17,4447.7,0.615
2,6447bd2acb4eeecd9451dbbe,2,20,200,1904,17,4162.6,0.750
3,67f789ca382e36a759a011af,1,3,30,254,1,6318.5,0.567
4,698354c40e8075e4c4602fbf,1,20,200,2116,17,4398.6,0.590
5,662121d5f125febc8db80f74,1,20,200,2456,17,4634.4,0.565
6,69a5dfa3f587ca27e1caf19d,2,20,200,1224,17,5710.6,0.650
7,6932ba422722a936d9dc7159,1,20,200,1933,17,5780.4,0.625
8,6a33b6f5e3d9dff938626d9d,1,20,200,857,17,3802.1,0.735
9,6a9c95b942aecb1a5e16ca69,1,20,200,1933,17,5269.6,0.700


In [17]:
# Die CSV enthaelt keine Spalte finalParameterSet (auch die aufbereiteten JSONs haben dort null).
# Indirekt erkennbar: Das finale Modell wird erst nach Block 18 berechnet (phase "final-model" in nextParameterSet)
# und muss dem nicht-Android-Block unter Block 19/20 entsprechen.
ANDROID_PARAMS = {"scrollFriction": 0.015, "decelerationRate": 2.3582, "inflexion": 0.35}
PARAM_TOL = {"scrollFriction": 1e-3, "decelerationRate": 1e-2, "inflexion": 1e-2}


def same_params(a, b):
    return all(abs(a[k] - b[k]) <= tol for k, tol in PARAM_TOL.items())


rows = []
for _, r in df.iterrows():
    blocks = sorted(r["blocks"], key=lambda b: b["runNumber"])
    if not blocks:
        continue
    nxt = dynamo_to_py({"M": json.loads(r["nextParameterSet"])}) if pd.notna(r["nextParameterSet"]) else None
    phase = ((nxt or {}).get("generation") or {}).get("phase")
    final_blocks = [b for b in blocks if b["runNumber"] >= 19]
    android = [b["runNumber"] for b in final_blocks if same_params(b["parameterSet"], ANDROID_PARAMS)]
    optimized = [b for b in final_blocks if not same_params(b["parameterSet"], ANDROID_PARAMS)]
    rows.append({
        "prolificPid": r["prolificPid"],
        "id": r["id"],
        "group": r["group"],
        "n_blocks": len(blocks),
        "next_phase": phase,
        "next_completedBlockCount": (nxt or {}).get("completedBlockCount"),
        "final_model_generated": phase == "final-model",
        "android_block": android[0] if android else None,
        "optimized_block": optimized[0]["runNumber"] if optimized else None,
        "optimized_matches_next": same_params(optimized[0]["parameterSet"], nxt) if optimized and nxt else None,
    })

final_check = pd.DataFrame(rows)

print("Eintraege mit Versuchen:", len(final_check))
print("\nPhase der nextParameterSet:")
print(final_check["next_phase"].value_counts(dropna=False).to_string())
print("\nFinales Modell berechnet (final-model):", final_check["final_model_generated"].sum())
print("Davon mit 20 Bloecken:", (final_check["final_model_generated"] & (final_check["n_blocks"] >= REQUIRED_BLOCKS)).sum())
print("Final-Phase erreicht (>= 18 Bloecke), aber kein final-model:", ((final_check["n_blocks"] >= 18) & ~final_check["final_model_generated"]).sum())
print("Optimierter Block 19/20 stimmt mit nextParameterSet ueberein:", final_check["optimized_matches_next"].eq(True).sum(), "von", final_check["optimized_matches_next"].notna().sum())

final_check[final_check["n_blocks"] >= 18].sort_values("n_blocks", ascending=False)

Eintraege mit Versuchen: 38

Phase der nextParameterSet:
next_phase
adaptive-qnei    23
final-model      12
NaN               3

Finales Modell berechnet (final-model): 12
Davon mit 20 Bloecken: 10
Final-Phase erreicht (>= 18 Bloecke), aber kein final-model: 0
Optimierter Block 19/20 stimmt mit nextParameterSet ueberein: 10 von 10


,prolificPid,id,group,n_blocks,next_phase,next_completedBlockCount,final_model_generated,android_block,optimized_block,optimized_matches_next
0,6932ba422722a936d9dc7159,b0b63c19-2a58-4d5f-b847-b34b881a46fb,1,20,final-model,19.0,True,20.0,19.0,True
3,69a5dfa3f587ca27e1caf19d,7194ce45-f67c-4e35-b2fa-dec79f6f500f,2,20,final-model,19.0,True,19.0,20.0,True
4,698a8b3d9811cfd9e748450e,f2003946-965e-4a66-a438-60734e99ccb2,2,20,final-model,19.0,True,19.0,20.0,True
5,6a9c95b942aecb1a5e16ca69,050f3f9b-f699-4175-a200-6378fba85425,1,20,final-model,19.0,True,20.0,19.0,True
6,662121d5f125febc8db80f74,5fdb6967-2279-4113-ba9e-4643015d1858,1,20,final-model,19.0,True,20.0,19.0,True
7,6a33b6f5e3d9dff938626d9d,313de046-9eb8-4fd6-afcc-4f3ca00fdad1,1,20,final-model,19.0,True,20.0,19.0,True
8,698354c40e8075e4c4602fbf,2384fbcc-826c-402f-8069-08f92495bea6,1,20,final-model,19.0,True,20.0,19.0,True
10,6a29409ecb14cd57b7d98746,c0b77808-319e-4595-99ea-8b0627ce3f2e,1,20,final-model,19.0,True,20.0,19.0,True
13,6447bd2acb4eeecd9451dbbe,c1404ef8-6b67-4e8e-8060-fb93d9ca7187,2,20,final-model,19.0,True,19.0,20.0,True
20,6a968836c2a369c9285714da,ea07ded7-91ef-401c-910a-94a66cfab9e2,1,20,final-model,19.0,True,20.0,19.0,True


In [18]:
ERGEBNISSE_DIR = CSV_PATH.parent.parent / "ergebnisse"  # statistische_auswertung.ipynb liest nur *.json direkt aus diesem Ordner
START_INDEX = 1

# statistische_auswertung.ipynb: Android-Block wird an diesen Default-Parametern erkannt
ANDROID_DEFAULTS = {"scrollFriction": 0.015, "decelerationRate": math.log(0.78) / math.log(0.9), "inflexion": 0.35}
ANDROID_TOL = {"scrollFriction": 1e-3, "decelerationRate": 1e-2, "inflexion": 1e-2}


def is_android(parameter_set):
    return all(abs(parameter_set[k] - ANDROID_DEFAULTS[k]) <= tol for k, tol in ANDROID_TOL.items())


# Eintraege, die Block 19 und 20 enthalten; pro Prolific-ID der mit den meisten Bloecken (dann juengster)
has_both = df["blocks"].apply(lambda bs: {19, 20} <= {b["runNumber"] for b in bs})
candidates = (
    df[has_both]
    .assign(n_blocks=lambda d: d["blocks"].apply(len))
    .sort_values(["n_blocks", "updatedAt"], ascending=False)
    .drop_duplicates("prolificPid")
    .sort_values("createdAt")
)

# Fruehere, von dieser Zelle erzeugte Dateien (Dateiname mit 24-stelliger Prolific-ID) entfernen, damit nichts doppelt gezaehlt wird
for old_file in ERGEBNISSE_DIR.glob("touch-scrolling-data-p*-*.json"):
    if re.fullmatch(r"touch-scrolling-data-p\d+-[0-9a-f]{24}-\d{4}-\d{2}-\d{2}\.json", old_file.name):
        old_file.unlink()

export_rows = []
for index, (_, row) in enumerate(candidates.iterrows(), start=START_INDEX):
    participant = prepare_participant(row.to_dict())
    file_name = f"touch-scrolling-data-p{index}-{row['prolificPid']}-{row['createdAt'][:10]}.json"
    with open(ERGEBNISSE_DIR / file_name, "w", encoding="utf-8") as fh:
        json.dump(participant, fh, ensure_ascii=False, indent=2)

    # Kontrolle wie in der statistischen Auswertung: die beiden letzten Bloecke muessen genau einmal Android enthalten
    with open(ERGEBNISSE_DIR / file_name, encoding="utf-8") as fh:
        last_two = sorted(json.load(fh)["attemptBlocks"], key=lambda b: b["runNumber"])[-2:]
    android_runs = [b["runNumber"] for b in last_two if is_android(b["parameterSet"])]
    export_rows.append({
        "file": file_name,
        "gender": participant["gender"],
        "group": participant["group"],
        "blocks": len(participant["attemptBlocks"]),
        "attempts": len(participant["attempts"]),
        "android_block": android_runs[0] if len(android_runs) == 1 else None,
        "stats_ready": len(last_two) == 2 and len(android_runs) == 1,
    })

export_overview = pd.DataFrame(export_rows)
print(f"{len(export_overview)} Dateien nach {ERGEBNISSE_DIR} geschrieben; "
      f"fuer statistische_auswertung.ipynb nutzbar: {int(export_overview['stats_ready'].sum())}")
export_overview

10 Dateien nach C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\ergebnisse geschrieben; fuer statistische_auswertung.ipynb nutzbar: 10


,file,gender,group,blocks,attempts,android_block,stats_ready
0,touch-scrolling-data-p1-6a968836c2a369c9285714...,female,1,20,200,20,True
1,touch-scrolling-data-p2-6447bd2acb4eeecd9451db...,female,2,20,200,19,True
2,touch-scrolling-data-p3-662121d5f125febc8db80f...,male,1,20,200,20,True
3,touch-scrolling-data-p4-6a29409ecb14cd57b7d987...,female,1,20,200,20,True
4,touch-scrolling-data-p5-698354c40e8075e4c4602f...,male,1,20,200,20,True
5,touch-scrolling-data-p6-698a8b3d9811cfd9e74845...,female,2,20,200,19,True
6,touch-scrolling-data-p7-69a5dfa3f587ca27e1caf1...,male,2,20,200,19,True
7,touch-scrolling-data-p8-6a33b6f5e3d9dff938626d...,male,1,20,200,20,True
8,touch-scrolling-data-p9-6932ba422722a936d9dc71...,male,1,20,200,20,True
9,touch-scrolling-data-p10-6a9c95b942aecb1a5e16c...,male,1,20,200,20,True
